# 02. 월간 품질시험·검사 실적보고서

**실시대장을 세기만 하면 보고서가 나옵니다.** 손으로 세지 않습니다.

숫자가 이상하면 대장이 이상한 것입니다. 아래에서 출처(대장!시트!행)를
같이 보여 주니 바로 확인할 수 있습니다.

> 집계는 **읽기 전용**입니다. Excel 이 없어도 됩니다.

In [ ]:
# 이 셀을 먼저 실행하세요. 어디서 열어도 프로젝트를 찾습니다.
import sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent                      # notebooks/ 에서 열었을 때
assert (ROOT / "main.py").exists(), f"프로젝트를 찾지 못했습니다: {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

def 표(머리, 행들):
    """리스트를 표로 보여 준다."""
    if not 행들:
        display(Markdown("_내용 없음_"))
        return
    md = "| " + " | ".join(str(h) for h in 머리) + " |\n"
    md += "|" + "|".join("---" for _ in 머리) + "|\n"
    for r in 행들:
        md += "| " + " | ".join("" if c is None else str(c) for c in r) + " |\n"
    display(Markdown(md))

from core.config import load_config

# 설정은 현재 폴더 -> 프로젝트 폴더 순으로 찾고, 없으면 예시에서 만들어 준다
후보 = [pathlib.Path.cwd() / "config.yaml", ROOT / "config.yaml"]
cfg = load_config(next((p for p in 후보 if p.exists()), None))
print("프로젝트:", ROOT)
print("설정 파일:", cfg.source)

## 1. 대장에 어떤 달이 있나

In [ ]:
from core.monthly_report import MonthlyReporter

reporter = MonthlyReporter(cfg)
달들 = reporter.있는_달()
print("대장에 있는 달:", ", ".join(달들) if 달들 else "없음 (경로를 확인하세요)")

## 2. 집계할 달 고르기

아래 `월` 을 바꾸고 실행하세요.

In [ ]:
월 = 달들[-1] if 달들 else "26.08"      # 예: "26.08"
print("집계할 달:", 월)

## 3. 집계

In [ ]:
전월, 금월 = reporter.누계(월)
누계 = 전월.더하기(금월)

표(["종목", "금월 실시", "합격", "불합격", "재시험", "누계 실시"],
   [(이름, v.실시, v.합격, v.불합격, v.재시험,
     누계.종목[이름].실시 if 이름 in 누계.종목 else "")
    for 이름, v in sorted(금월.종목.items())])

print(f"금월 {금월.총건수}건 · 전월까지 {전월.총건수}건 · 누계 {누계.총건수}건")

## 4. 확인 필요

**'세지 못했습니다'** 가 뜨면 그 시험이 보고서에서 빠집니다.
`templates/실적보고서/종목매핑.yaml` 에 규칙을 추가하세요.

In [ ]:
경고 = [*dict.fromkeys([*전월.경고, *금월.경고])]
if 경고:
    for w in 경고:
        print(" -", w)
else:
    print("이상 없음")

## 5. 이 숫자가 어디서 나왔나

`대장!시트!행` 입니다. 엑셀에서 그 행을 바로 열어 확인할 수 있습니다.

In [ ]:
표(["종목", "건수", "출처"],
   [(이름, v.실시, ", ".join(v.출처)) for 이름, v in sorted(금월.종목.items())])

## 6. 분기 누계 (성과총괄표용)

In [ ]:
연 = 2000 + int(월.split(".")[0])
분기번호 = (int(월.split(".")[1]) - 1) // 3 + 1
q = reporter.분기(연, 분기번호)
print(f"{연}년 {분기번호}분기")
표(["종목", "실시", "합격"], [(k, v.실시, v.합격) for k, v in sorted(q.종목.items())])

## 7. 보고서 파일에 쓰기

여기서부터는 **파일을 고칩니다.** Excel 이 설치된 Windows 에서만 됩니다.

보고서 파일은 셀 좌표를 실측하지 못해 글자로 자리를 찾습니다.
못 찾으면 아무 데나 쓰지 않고 멈춥니다.

아래 `쓰기 = True` 로 바꿔야 실제로 실행됩니다.

In [ ]:
쓰기 = False        # <- True 로 바꾸면 실제로 보고서를 고칩니다

from tasks.monthly_report import MonthlyInput, MonthlyReportTask

data = MonthlyInput(월=월, 전월=전월, 금월=금월)
task = MonthlyReportTask(cfg)

문제 = task.preview(data)
print(문제.메시지)
for w in 문제.경고:
    print(" -", w)

if 쓰기:
    결과 = task.run(data)
    print("\n실행:", 결과.메시지)
    for w in 결과.경고:
        print(" -", w)
else:
    print("\n(쓰기 = False 라 실제로는 쓰지 않았습니다)")